# CT 촬영 품질 게이트 — MobileNetV3-Small

YOLO 결함탐지 **앞단**에서 "이 CT 사진이 검사에 쓸 만큼 잘 찍혔나"만 판별하는 이진 분류기입니다.
결함 유무(정상/불량 배터리)는 보지 않습니다. **오직 촬영 품질**만 봅니다.

```
CT 사진 → [품질 게이트] → PASS → YOLO 결함탐지로 진행
                       → FAIL → 재촬영
```

## 판별 대상 — 촬영 실패 5종

| 실패 케이스 | 무엇이 잘못된 사진인가 |
|---|---|
| `ct_cell_alignment_failure` | FOV 정렬 실패로 배터리가 촬영 범위 밖에서 잘림 |
| `ct_acquisition_motion` | 촬영 중 움직여 이중 영상이 겹침 |
| `ct_insufficient_projection_sampling` | 투영 수·각도 부족 → 줄무늬 + 미세구조 소실 |
| `ct_low_signal_noise` | 관전류·노출 부족 → 어둡고 노이즈 심함 |
| `ct_beam_hardening_metal_streak` | 금속 주변 beam hardening → 방사형 streak |

## 이 노트북의 세 가지 설계 원칙

**1. 분할은 battery_id 단위로 한다**
제공된 `main`/`test` 폴더는 배터리가 100% 겹칩니다. 그대로 쓰면 성능이 부풀려집니다(측정값: 엄격 운영점에서 **+8.3%p**).
CT는 배터리 1개에서 평균 425장의 인접 슬라이스가 나오므로 특히 취약합니다.

**2. accuracy를 쓰지 않는다**
PASS:FAIL = 90:10이라 "전부 PASS"라고만 찍어도 90%가 나옵니다.
**fail recall**(FAIL을 잡는 비율)과 **clean FPR**(정상을 FAIL로 오판 = 재촬영 = 수율 손실)을 따로 봅니다.

**3. crop 없이 letterbox만 쓴다**
잘라내면 "배터리가 화면 밖으로 나갔다"는 FAIL의 증거가 함께 잘려나갑니다.

---
## 0. 환경 및 설정

PyTorch CPU 환경에서 동작합니다(GPU 불필요).

```
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install numpy pillow pandas scikit-learn matplotlib
```

In [ ]:
import csv, io, json, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision as tv
import matplotlib.pyplot as plt
from PIL import Image

# ── 설정 ────────────────────────────────────────────────────────────
DATA_ROOT = Path(r"C:\quality_fail_40k_v1.8_20260730_unzip\full_images")  # 원본 (읽기 전용)
OUT       = Path("./ct_quality_gate_out")                                  # 산출물
MODALITY  = "CT"
CANVAS    = (288, 512)      # (W, H) — 아래 2절에서 실측으로 근거 제시
PAD       = 114             # letterbox 여백값. PASS·FAIL 동일하므로 shortcut 안 됨
SEED      = 20260731

N_FOLDS   = 5
N_LOCKBOX = 9               # 최종 확인용 봉인 셀 수

BATCH     = 48
EPOCHS    = 4
LR        = 3e-4
POS_W     = 9.0             # PASS:FAIL = 90:10 → 양성(FAIL) 가중

FPR_TARGETS = (0.01, 0.03, 0.05)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

OUT.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
print("torch", torch.__version__, "| threads", torch.get_num_threads())

---
## 1. 데이터 인덱스 — 라벨 3중 대조

라벨의 정본은 `labels_json/*.json`의 최상위 `quality_class`입니다.
`pass`/`fail` 폴더, `augmentation_json` 존재 여부와 **세 곳을 대조**해 불일치가 없는지 확인합니다.

증강 이력 JSON에는 어떤 실패 유형인지(`failure_case`)와 **얼마나 세게 망가뜨렸는지**(파라미터)가
기록돼 있습니다. `failure_case`는 학습에 쓰지 않고 **케이스별 성능 보고**에만 씁니다.

In [ ]:
def parse_stem(stem: str) -> dict:
    '''CT_cell_<form>_<battery_id>_<axis>_<image_id>'''
    p = stem.split("_")
    return {"form": p[2], "battery_id": p[3], "axis": p[4], "image_id": p[5]}


def build_index() -> list[dict]:
    rows, problems = [], []
    for partition in ("main", "test"):
        base = DATA_ROOT / MODALITY / partition

        # 증강 이력 → stem별 failure_case
        aug = {}
        for f in (base / "augmentation_json").glob("*.json"):
            stem = f.name.split(".augmentation")[0]
            aug[stem] = json.loads(f.read_text(encoding="utf-8"))["failure_case"]["id"]

        folder_fail = {p.stem for p in (base / "fail").glob("*.jpg")}
        folder_pass = {p.stem for p in (base / "pass").glob("*.jpg")}

        for img in sorted((base / "images").glob("*.jpg")):
            stem = img.stem
            lf = base / "labels_json" / f"{stem}.json"
            if not lf.exists():
                problems.append(f"라벨 JSON 없음: {stem}"); continue
            qc = json.loads(lf.read_text(encoding="utf-8"))["quality_class"]

            by_folder = "fail" if stem in folder_fail else ("pass" if stem in folder_pass else None)
            by_aug = "fail" if stem in aug else "pass"
            if by_folder is not None and by_folder != qc:
                problems.append(f"폴더 불일치: {stem}")
            if by_aug != qc:
                problems.append(f"증강이력 불일치: {stem}")

            row = {"stem": stem, "quality_class": qc, "failure_case": aug.get(stem, ""),
                   "image_path": str(img), "orig_partition": partition}
            row.update(parse_stem(stem))
            rows.append(row)

    print(f"3중 대조 불일치: {len(problems)}건" + ("" if not problems else f" → {problems[:5]}"))
    return rows


INDEX = build_index()
print(f"총 {len(INDEX)}행 | 고유 battery {len({r['battery_id'] for r in INDEX})}개")
print("quality_class:", dict(Counter(r["quality_class"] for r in INDEX)))
print("failure_case :", dict(Counter(r["failure_case"] for r in INDEX if r["failure_case"])))

---
## 2. CT 형상 실측 — 입력 캔버스를 정하는 근거

CT는 ROI crop 결과라 **높이는 전부 512로 고정**이고 **폭만 46~282**로 변합니다.

정사각 512×512로 letterbox하면 **평균 74%가 여백**이 됩니다.
최대 폭이 282이므로 **288×512면 화소를 하나도 잃지 않으면서 연산량이 0.56배**입니다.
MobileNetV3는 끝단이 adaptive pool이라 비정사각 입력을 그대로 받습니다.

또한 **종횡비가 shortcut이 되는지** 확인합니다.
`ct_cell_alignment_failure`는 crop 후 원본 raster 크기로 되돌리므로 종횡비가 보존돼야 합니다.

In [ ]:
def measure_geometry(rows, sample=None):
    rs = rows if sample is None else rows[:: max(1, len(rows) // sample)]
    ws, hs = [], Counter()
    by_axis, by_class = defaultdict(list), defaultdict(list)
    for r in rs:
        with Image.open(r["image_path"]) as im:
            w, h = im.size
        ws.append(w); hs[h] += 1
        by_axis[r["axis"]].append(w)
        by_class[r["quality_class"]].append(w / h)

    ws.sort()
    print(f"height 분포: {dict(hs)}")
    print(f"width  min={ws[0]}  p50={ws[len(ws)//2]}  p99={ws[int(.99*len(ws))]}  max={ws[-1]}")
    for ax in sorted(by_axis):
        v = sorted(by_axis[ax])
        print(f"  axis {ax}: n={len(v):>5}  min={v[0]:>4}  p50={v[len(v)//2]:>4}  max={v[-1]:>4}")

    print("\n종횡비(w/h) — PASS vs FAIL (shortcut 점검)")
    for k in ("pass", "fail"):
        v = sorted(by_class[k])
        print(f"  {k:<5} p5={v[int(.05*len(v))]:.4f}  p50={v[len(v)//2]:.4f}  p95={v[int(.95*len(v))]:.4f}")

    print("\n캔버스 폭별 비용")
    for cw in (192, 256, 288, 512):
        cover = sum(1 for w in ws if w <= cw) / len(ws)
        print(f"  {cw:>3}: 무손실 커버 {cover:6.1%}   픽셀수 {cw*512/(512*512):.2f}× (정사각 대비)")


measure_geometry(INDEX, sample=4000)   # 전수는 sample=None

> **읽는 법**
> PASS와 FAIL의 종횡비 분포가 사실상 같으면(p50 동일) 모델이 종횡비만 보고 맞히는 경로가 없다는 뜻입니다.
> 실측 결과 두 분포의 p50이 모두 0.2891로 동일했습니다.

---
## 3. ⚠️ battery_id 단위 재분할 — 이 노트북에서 가장 중요한 부분

**제공된 `main`/`test` 폴더를 그대로 쓰면 안 됩니다.**
test의 배터리 47개가 **100% 전부 main에도** 들어 있습니다.
증강계획서가 *"main/test는 `quality_class`만 90:10으로 맞추며 원본 split은 층화하지 않는다"* 고
명시한 대로의 결과라 **사양 위반은 아니지만**, 학습·평가용으로는 쓸 수 없습니다.

CT는 배터리 1개에서 평균 425장의 인접 슬라이스가 나옵니다.
옆 단면은 거의 같은 사진이라, 모델이 "촬영 품질"이 아니라 **"이 셀은 이렇게 생겼다"** 를 외워서 맞힙니다.

**통제 실험으로 측정한 부풀림**(같은 학습셋·같은 모델, 시험지만 교체):

| 시험지 | PR-AUC | recall@FPR1% | recall@FPR3% |
|---|---|---|---|
| 같은 셀 (누수) | 0.9973 | 0.9896 | 1.0000 |
| 다른 셀 (정직) | 0.9759 | 0.9065 | 0.9782 |
| **차이** | +0.0213 | **+8.3%p** | +2.2%p |

그래서 아래처럼 **셀 단위로 다시 나눕니다.**
독립 표본이 배터리 47개뿐이라 단일 split을 믿지 않고 **5-fold + lockbox 봉인** 구조로 갑니다.

In [ ]:
def battery_stats(rows):
    st = defaultdict(lambda: {"n": 0, "n_fail": 0})
    for r in rows:
        s = st[r["battery_id"]]
        s["n"] += 1
        s["n_fail"] += (r["quality_class"] == "fail")
    return st


def make_splits(rows, n_folds=N_FOLDS, n_lockbox=N_LOCKBOX, seed=SEED):
    '''lockbox를 봉인하고 나머지를 FAIL 장수 기준 그리디로 n_folds 등분.'''
    import random
    st = battery_stats(rows)
    rng = random.Random(seed)
    bids = sorted(st, key=lambda b: (st[b]["n_fail"], b))

    # 분포 대표성을 유지하며 균등 간격으로 lockbox 추출
    step = len(bids) / n_lockbox
    lock = {bids[min(int(i * step + step / 2), len(bids) - 1)] for i in range(n_lockbox)}

    rest = [b for b in bids if b not in lock]
    rng.shuffle(rest)
    rest.sort(key=lambda b: st[b]["n_fail"], reverse=True)

    fold_fail = [0] * n_folds
    assign = {}
    for b in rest:
        f = min(range(n_folds), key=lambda i: fold_fail[i])
        assign[b] = f"fold{f}"
        fold_fail[f] += st[b]["n_fail"]
    for b in lock:
        assign[b] = "lockbox"
    return assign


SPLIT_OF_BID = make_splits(INDEX)
for r in INDEX:
    r["split"] = SPLIT_OF_BID[r["battery_id"]]

# 검증: split 간 battery 교집합이 0이어야 한다
by_split = defaultdict(list)
for r in INDEX:
    by_split[r["split"]].append(r)

print(f"{'split':<9}{'셀':>5}{'이미지':>9}{'FAIL':>7}{'FAIL비율':>10}")
cells = {}
for k in sorted(by_split):
    v = by_split[k]
    cells[k] = {r["battery_id"] for r in v}
    nf = sum(r["quality_class"] == "fail" for r in v)
    print(f"{k:<9}{len(cells[k]):>5}{len(v):>9}{nf:>7}{100*nf/len(v):>9.1f}%")

ks = sorted(cells)
overlap = sum(len(cells[a] & cells[b]) for i, a in enumerate(ks) for b in ks[i+1:])
print(f"\nsplit 간 battery 교집합: {overlap}개  (0이어야 정상)")
print("lockbox는 모든 개발이 끝난 뒤 단 1회만 개봉합니다.")

---
## 4. letterbox 전처리 캐시

**crop은 절대 하지 않습니다.** 종횡비를 보존한 채 축소하고 남는 곳을 균일한 회색(114)으로 채웁니다.
여백값은 PASS·FAIL에 동일하게 적용되므로 판별 shortcut이 되지 않습니다.

CPU 학습에서 매 epoch JPEG 디코드를 반복하지 않도록 결과를 uint8 memmap 한 덩어리로 만들어 둡니다.
20,000장 × 288×512×3 ≈ **8.8 GB**, 생성에 약 30초 걸립니다.

In [ ]:
def letterbox(img: Image.Image, canvas=CANVAS) -> Image.Image:
    cw, ch = canvas
    w, h = img.size
    s = min(cw / w, ch / h)                       # 확대하지 않음
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    img = img.resize((nw, nh), Image.BILINEAR)
    out = Image.new(img.mode, (cw, ch), (PAD,) * len(img.getbands()))
    out.paste(img, ((cw - nw) // 2, (ch - nh) // 2))
    return out


def build_cache(rows, canvas=CANVAS, force=False):
    cw, ch = canvas
    path = OUT / f"{MODALITY}_{cw}x{ch}.npy"
    if path.exists() and not force:
        print(f"캐시 재사용: {path}")
        return np.load(path, mmap_mode="r")

    n = len(rows)
    print(f"캐시 생성: {path}  ({n * ch * cw * 3 / 1024**3:.1f} GB)")
    arr = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=(n, ch, cw, 3))
    t0 = time.time()
    for i, r in enumerate(rows):
        with Image.open(r["image_path"]) as im:
            # CT는 흑백이므로 L로 읽은 뒤 3채널 복제 → ImageNet 사전학습 가중치를 그대로 사용
            arr[i] = np.asarray(letterbox(im.convert("L"), canvas).convert("RGB"), dtype=np.uint8)
        if (i + 1) % 4000 == 0:
            print(f"  {i+1}/{n}  {time.time()-t0:.0f}s")
    arr.flush()
    print(f"  완료 {time.time()-t0:.0f}s")
    return np.load(path, mmap_mode="r")


CACHE = build_cache(INDEX)
Y      = np.array([r["quality_class"] == "fail" for r in INDEX], dtype=np.float32)
CASES  = np.array([r["failure_case"] for r in INDEX])
SPLITS = np.array([r["split"] for r in INDEX])
print("캐시 shape:", CACHE.shape)

In [ ]:
# letterbox 결과 눈으로 확인
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
show = list(np.where(Y == 0)[0][:3]) + list(np.where(Y == 1)[0][:3])
for ax, i in zip(axes, show):
    ax.imshow(CACHE[i]); ax.axis("off")
    ax.set_title(f"{'FAIL' if Y[i] else 'PASS'}\n{CASES[i][3:] or '-'}", fontsize=8)
plt.tight_layout(); plt.show()

---
## 5. 모델 — MobileNetV3-Small + 이진 head

ImageNet 사전학습 가중치를 불러와 마지막 층만 `Linear(1024, 1)`로 교체합니다(1-logit 이진 판정).
backbone 동결 없이 전체 미세조정합니다.

**학습 증강은 flip만 씁니다.** 밝기·블러·노이즈 증강은 **그 자체가 FAIL 신호**라
PASS에 걸면 라벨이 오염됩니다. CT ROI는 상하·좌우 대칭이 물리적으로 유효하므로 hflip + vflip만 사용합니다.

In [ ]:
def build_model() -> nn.Module:
    w = tv.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
    m = tv.models.mobilenet_v3_small(weights=w)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    return m


def to_batch(idx: np.ndarray, train: bool) -> torch.Tensor:
    x = torch.from_numpy(np.ascontiguousarray(CACHE[idx]))      # [B,H,W,3] uint8
    x = x.permute(0, 3, 1, 2).float().div_(255.0)
    if train:
        f = torch.rand(x.shape[0]) < 0.5
        x[f] = torch.flip(x[f], dims=[3])                       # 좌우
        f = torch.rand(x.shape[0]) < 0.5
        x[f] = torch.flip(x[f], dims=[2])                       # 상하
    return (x - IMAGENET_MEAN) / IMAGENET_STD


m = build_model()
print(f"파라미터 {sum(p.numel() for p in m.parameters()):,}개")
print("classifier:", m.classifier)

---
## 6. 지표 — accuracy를 쓰지 않는 이유

PASS:FAIL = 90:10이라 **"전부 PASS"라고만 찍는 모델도 정확도 90%** 가 나옵니다.
대신 두 축을 **따로** 봅니다.

| 지표 | 뜻 | 나빠지면 |
|---|---|---|
| **fail recall** | 실제 FAIL 중 몇 %를 잡았나 | 나쁜 사진이 YOLO로 흘러감 |
| **clean FPR** | 정상 중 몇 %를 FAIL로 오판했나 | 멀쩡한 셀 재촬영 = 수율 손실 |

둘은 맞바꾸는 관계이므로 **clean FPR 상한을 먼저 정하고 그 안에서 recall 최대**인 지점을 운영점으로 씁니다.
상한이 확정되기 전이라 1% / 3% / 5% 세 지점을 모두 출력합니다.

In [ ]:
def recall_at_fpr(y, score, target_fpr):
    '''clean FPR 상한을 지키는 threshold에서의 fail recall → (recall, threshold)'''
    neg = np.sort(score[y == 0])[::-1]
    k = int(np.floor(target_fpr * len(neg)))
    thr = np.nextafter(neg[k - 1] if k > 0 else neg[0], np.inf)
    return float((score[y == 1] >= thr).mean()), float(thr)


def pr_auc(y, score):
    o = np.argsort(-score); y = y[o]
    tp, fp = np.cumsum(y), np.cumsum(1 - y)
    prec = tp / np.maximum(tp + fp, 1)
    rec = tp / max(y.sum(), 1)
    return float(np.sum(np.diff(np.concatenate([[0.0], rec])) * prec))


def summarize(y, score, cases=None) -> dict:
    out = {"n": int(len(y)), "n_fail": int(y.sum()), "pr_auc": pr_auc(y, score)}
    for t in FPR_TARGETS:
        r, thr = recall_at_fpr(y, score, t)
        out[f"recall@fpr{t:.0%}"] = r
        out[f"thr@fpr{t:.0%}"] = thr
    if cases is not None:
        thr = out["thr@fpr3%"]
        out["per_case"] = {c: (float((score[(y == 1) & (cases == c)] >= thr).mean()),
                               int(((y == 1) & (cases == c)).sum()))
                           for c in sorted({c for c in cases[y == 1] if c})}
    return out


def show_metrics(m, title=""):
    print(f"[{title}] n={m['n']}  fail={m['n_fail']}  PR-AUC={m['pr_auc']:.4f}")
    for t in FPR_TARGETS:
        print(f"  fail recall @ clean FPR {t:.0%} = {m[f'recall@fpr{t:.0%}']:.4f}")
    if "per_case" in m:
        print("  케이스별 recall @ FPR 3%")
        for c, (r, n) in m["per_case"].items():
            print(f"    {c:<38}{r:.3f}  (n={n})")

---
## 7. 학습

`fold k`를 시험지로 쓸 때 **train = 나머지 3개 fold, val = fold k+1** 입니다.
`lockbox`는 어디에도 들어가지 않습니다.

CPU 기준 epoch당 8~9분입니다. 매 epoch 끝에 val 성적이 가장 좋은 상태를 저장하므로
중간에 끊어도 그때까지의 최선이 남습니다.

In [ ]:
def fold_splits(fold: int):
    test = f"fold{fold % N_FOLDS}"
    val  = f"fold{(fold + 1) % N_FOLDS}"
    train = [f"fold{i}" for i in range(N_FOLDS) if f"fold{i}" not in (test, val)]
    return train, val, test


@torch.no_grad()
def evaluate(model, idx, bs=BATCH):
    model.eval()
    out = np.empty(len(idx), dtype=np.float32)
    for i in range(0, len(idx), bs):
        sl = idx[i:i + bs]
        out[i:i + len(sl)] = model(to_batch(sl, False)).squeeze(1).numpy()
    return out


def train_fold(fold=0, epochs=EPOCHS, exclude_case=""):
    tr_names, va_name, te_name = fold_splits(fold)
    idx_tr = np.where(np.isin(SPLITS, tr_names))[0]
    idx_va = np.where(SPLITS == va_name)[0]
    idx_te = np.where(SPLITS == te_name)[0]

    if exclude_case:   # leave-one-case-out 용 (9절)
        drop = (CASES == exclude_case) & (Y == 1)
        idx_tr = idx_tr[~drop[idx_tr]]

    run = OUT / f"fold{fold}" / (f"no_{exclude_case}" if exclude_case else "base")
    run.mkdir(parents=True, exist_ok=True)
    print(f"fold{fold}: train={'+'.join(tr_names)} val={va_name} test={te_name}")
    print(f"  train {len(idx_tr)}장  val {len(idx_va)}장  test {len(idx_te)}장  (lockbox 제외)")

    model = build_model()
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_W]))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    steps = max(1, len(idx_tr) // BATCH)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps)

    best, hist = -1.0, []
    for ep in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(idx_tr)
        t0, tot = time.time(), 0.0
        for i in range(0, steps * BATCH, BATCH):
            sl = np.sort(perm[i:i + BATCH])
            loss = crit(model(to_batch(sl, True)).squeeze(1), torch.from_numpy(Y[sl]))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        mv = summarize(Y[idx_va], evaluate(model, idx_va))
        key = mv["recall@fpr3%"]
        hist.append({"epoch": ep, "loss": tot / steps, "sec": time.time() - t0,
                     "val_pr_auc": mv["pr_auc"], "val_recall_fpr3": key})
        print(f"  ep{ep} loss {tot/steps:.4f}  {time.time()-t0:.0f}s  "
              f"val PR-AUC {mv['pr_auc']:.4f}  recall@FPR3% {key:.4f}")
        if key > best:
            best = key
            torch.save(model.state_dict(), run / "best.pt")

    model.load_state_dict(torch.load(run / "best.pt"))
    res = {"history": hist}
    for name, idx in (("val", idx_va), ("test", idx_te)):
        mm = summarize(Y[idx], evaluate(model, idx), CASES[idx])
        show_metrics(mm, f"{name.upper()} (학습에 쓰지 않은 셀)")
        res[name] = mm
    (run / "metrics.json").write_text(json.dumps(res, indent=2, ensure_ascii=False), encoding="utf-8")
    return model, res, (idx_tr, idx_va, idx_te)

In [ ]:
# 실행 (CPU 기준 epoch당 8~9분)
MODEL, RESULT, (IDX_TR, IDX_VA, IDX_TE) = train_fold(fold=0, epochs=EPOCHS)

In [ ]:
h = RESULT["history"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([e["epoch"] for e in h], [e["loss"] for e in h], marker="o")
ax[0].set_title("train loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)
ax[1].plot([e["epoch"] for e in h], [e["val_recall_fpr3"] for e in h], marker="o", label="recall@FPR3%")
ax[1].plot([e["epoch"] for e in h], [e["val_pr_auc"] for e in h], marker="s", label="PR-AUC")
ax[1].set_title("validation"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

---
## 8. 평가 — 운영점 선택

`clean FPR`을 x축, `fail recall`을 y축으로 놓고 **운영 정책에 맞는 지점**을 고릅니다.
"재촬영을 몇 %까지 감수할 수 있는가"가 정해지면 그 지점의 threshold만 쓰면 되고 **재학습은 불필요**합니다.

In [ ]:
SC_TEST = evaluate(MODEL, IDX_TE)
y_te = Y[IDX_TE]

# ── recall vs clean FPR 곡선 ──
fprs = np.linspace(0.001, 0.20, 120)
recs = [recall_at_fpr(y_te, SC_TEST, f)[0] for f in fprs]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(fprs * 100, recs)
for t in FPR_TARGETS:
    r, _ = recall_at_fpr(y_te, SC_TEST, t)
    ax[0].scatter([t * 100], [r], zorder=5)
    ax[0].annotate(f"{t:.0%}\n{r:.3f}", (t * 100, r), textcoords="offset points",
                   xytext=(6, -14), fontsize=8)
ax[0].set_xlabel("clean FPR (%)  ← 재촬영 비율"); ax[0].set_ylabel("fail recall")
ax[0].set_title("운영점 선택 곡선"); ax[0].grid(alpha=.3)

# ── 점수 분포 ──
ax[1].hist(SC_TEST[y_te == 0], bins=60, alpha=.6, label="PASS")
ax[1].hist(SC_TEST[y_te == 1], bins=60, alpha=.6, label="FAIL")
thr3 = RESULT["test"]["thr@fpr3%"]
ax[1].axvline(thr3, color="k", ls="--", label=f"운영점 (FPR 3%)")
ax[1].set_xlabel("logit"); ax[1].set_title("점수 분포"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# 운영점에서의 혼동행렬 — 주 지표는 아니고 현황 확인용
pred = (SC_TEST >= thr3).astype(int)
cm = np.array([[int(((pred == 0) & (y_te == 0)).sum()), int(((pred == 1) & (y_te == 0)).sum())],
               [int(((pred == 0) & (y_te == 1)).sum()), int(((pred == 1) & (y_te == 1)).sum())]])
print("               예측 PASS   예측 FAIL")
print(f"실제 PASS     {cm[0,0]:>9}   {cm[0,1]:>9}   ← 오른쪽이 재촬영 손실")
print(f"실제 FAIL     {cm[1,0]:>9}   {cm[1,1]:>9}   ← 왼쪽이 놓친 나쁜 사진")

# 케이스별 recall — "어떤 촬영 실패를 놓치는가"가 현장에서 가장 쓸모 있다
pc = RESULT["test"]["per_case"]
plt.figure(figsize=(8, 3))
plt.barh(list(pc.keys()), [v[0] for v in pc.values()])
plt.xlim(0, 1.05); plt.xlabel("recall @ clean FPR 3%"); plt.grid(alpha=.3, axis="x")
plt.title("케이스별 검출률"); plt.tight_layout(); plt.show()

---
## 9. (선택) leave-one-case-out — 처음 보는 실패 유형에도 통하는가

현재 데이터에는 5종만 있는데, 현장에서 6번째 유형이 나오지 말란 법이 없습니다.
직접 시험할 방법이 없으므로 **알고 있는 5종 중 하나를 "모르는 유형"인 척** 만들어 잽니다.

실측 결과(fold0, 운영점 FPR 3%):

| 제외한 케이스 | 학습 포함 시 | 제외 후 |
|---|---|---|
| `ct_beam_hardening_metal_streak` | 0.939 | **0.061** |
| `ct_acquisition_motion` | 0.952 | **0.587** |

**유형에 따라 편차가 큽니다.** 흐림·이중경계는 다른 케이스에서 전이되지만(0.587),
금속 주변 방사형 streak는 어떤 케이스와도 닮지 않아 거의 전이되지 않습니다(0.061).

→ **새 실패 유형 대응은 모델 선택으로 풀리지 않습니다.** 실촬영 데이터 확보가 필요합니다.

In [ ]:
# 케이스당 약 30분 소요. 필요할 때만 실행하세요.
RUN_LOCO = False

if RUN_LOCO:
    for c in ["ct_beam_hardening_metal_streak", "ct_acquisition_motion"]:
        print(f"\n===== {c} 제외하고 학습 =====")
        _, res_c, _ = train_fold(fold=0, epochs=EPOCHS, exclude_case=c)
        held = res_c["test"]["per_case"].get(c)
        base = RESULT["test"]["per_case"].get(c)
        print(f"{c}: 포함 {base[0]:.3f} → 제외 {held[0]:.3f}  ({held[0]-base[0]:+.3f})")

---
## 10. Grad-CAM — 모델이 어디를 보는가

단순히 PASS/FAIL을 판별하는 데 그치지 않고, **이미지의 어느 부분을 근거로 판단했는지** 확인합니다.
MobileNetV3-Small의 마지막 합성곱 층 활성값과 기울기를 곱해 히트맵을 만듭니다.

In [ ]:
def grad_cam(model, idx: int):
    model.eval()
    feats, grads = {}, {}
    target = model.features[-1]                    # 마지막 합성곱 블록
    h1 = target.register_forward_hook(lambda m, i, o: feats.__setitem__("v", o))
    h2 = target.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("v", go[0]))

    x = to_batch(np.array([idx]), False)
    logit = model(x).squeeze()
    model.zero_grad(); logit.backward()
    h1.remove(); h2.remove()

    w = grads["v"].mean(dim=(2, 3), keepdim=True)          # 채널별 가중치
    cam = torch.relu((w * feats["v"]).sum(1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                                          align_corners=False)[0, 0].detach().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, float(logit)


# FAIL 케이스별로 한 장씩 시각화
targets = []
for c in sorted({c for c in CASES[IDX_TE] if c}):
    cand = IDX_TE[(Y[IDX_TE] == 1) & (CASES[IDX_TE] == c)]
    if len(cand): targets.append((c, int(cand[0])))

fig, axes = plt.subplots(2, len(targets), figsize=(3 * len(targets), 8))
for col, (c, i) in enumerate(targets):
    cam, lg = grad_cam(MODEL, i)
    axes[0, col].imshow(CACHE[i]); axes[0, col].axis("off")
    axes[0, col].set_title(f"{c[3:]}\nlogit {lg:.1f}", fontsize=8)
    axes[1, col].imshow(CACHE[i]); axes[1, col].imshow(cam, cmap="jet", alpha=.45)
    axes[1, col].axis("off"); axes[1, col].set_title("Grad-CAM", fontsize=8)
plt.tight_layout(); plt.show()

---
## 11. 분류 실행 — 폴더를 넣고 PASS/FAIL 받기

운영점 threshold를 적용해 폴더 안 이미지를 일괄 판정하고 CSV로 저장합니다.
`fail_prob`는 눈으로 보기 편하라고 넣은 참고값입니다(**보정하지 않았으므로 확률로 해석하지 말 것**).
판정은 `logit`으로 합니다.

In [ ]:
@torch.no_grad()
def classify_folder(model, folder, threshold=None, out_csv="predictions.csv", bs=BATCH):
    thr = RESULT["val"]["thr@fpr3%"] if threshold is None else threshold
    paths = sorted(Path(folder).glob("*.jpg"))
    if not paths:
        raise SystemExit(f"이미지 없음: {folder}")
    model.eval()
    rows, n_fail = [], 0
    for i in range(0, len(paths), bs):
        chunk = paths[i:i + bs]
        arr = []
        for p in chunk:
            with Image.open(p) as im:
                arr.append(np.asarray(letterbox(im.convert("L")).convert("RGB"), dtype=np.uint8))
        x = torch.from_numpy(np.stack(arr)).permute(0, 3, 1, 2).float().div_(255.0)
        lg = model((x - IMAGENET_MEAN) / IMAGENET_STD).squeeze(1)
        for p, l, q in zip(chunk, lg.tolist(), torch.sigmoid(lg).tolist()):
            v = "FAIL" if l >= thr else "PASS"
            n_fail += v == "FAIL"
            rows.append({"file": p.name, "logit": round(l, 4),
                         "fail_prob": round(q, 4), "verdict": v})

    with open(out_csv, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=["file", "logit", "fail_prob", "verdict"]); w.writeheader()
        w.writerows(rows)
    print(f"운영점 threshold = {thr:.4f} (logit)")
    print(f"총 {len(rows)}장 → PASS {len(rows)-n_fail} / FAIL {n_fail} ({100*n_fail/len(rows):.1f}%)")
    print(f"결과 → {out_csv}")
    return rows


# 예시
# classify_folder(MODEL, r"C:\내\CT사진폴더", out_csv="결과.csv")

---
## 12. 결과 요약과 한계

### 성능 (fold0, 학습에 쓰지 않은 셀 7개 / 3,172장)

| 지표 | 값 |
|---|---|
| PR-AUC | 0.9896 |
| fail recall @ clean FPR 1% | 0.9564 |
| fail recall @ clean FPR 3% | 0.9782 |
| fail recall @ clean FPR 5% | 0.9938 |

### ⚠️ 이 숫자를 그대로 신뢰하면 안 되는 이유

FAIL 2,000장은 **전부 합성**이고 실제로 잘못 찍힌 사진은 **0장**입니다.

강도를 바꿔가며 확인한 결과, 모델은 증강 프로그램의 흔적이 아니라
**실제 물리적 열화에 반응**하는 것으로 나타났습니다(픽셀 무변화 재인코딩에서 점수 변화 +0.02).
다만 **데이터의 강도 구간이 전부 포화 영역**이라는 점이 확인됐습니다.

| | 모델이 반응하기 시작하는 지점 | 데이터에 들어 있는 범위 |
|---|---|---|
| 저신호 노이즈 | `photon_scale` 약 300~3,000 | 10 ~ 40 |
| 이중 영상 | offset 약 2px | 18 ~ 28px |

즉 **데이터에 애매한 사진이 하나도 없어서** 0.98이 나온 것입니다.
큰 화재 사진으로만 시험한 화재감지기가 100점을 받은 것과 같습니다.

### 남은 과제

1. **실환경 clean FPR 측정** — shadow 모드(판정을 기록만 하고 거르지 않음).
   정상 사진은 현장에 얼마든지 있으므로 실촬영 FAIL 없이도 측정 가능합니다.
2. **실촬영 실패 사진 확보** — 케이스당 20~30장이면 평가용으로 충분합니다.
3. **경계 강도 데이터** — 사양보다 약한 강도의 FAIL로 검출 문턱을 확정.
4. **lockbox 9셀** — 모든 개발이 끝난 뒤 단 1회만 개봉.

### 운영 권고

실환경 FPR이 확인되기 전까지는 **clean FPR 상한을 1% 이하로 보수적으로** 잡습니다.
recall이 다소 떨어져도 게이트가 없는 것보다 낫고 수율 피해가 없습니다.
`thr@fpr1%`를 쓰면 되며 **재학습은 불필요**합니다.